# Aula 5 — Montagem de novo com SPAdes

A montagem começa com os reads paired que sobreviveram ao processamento da Aula 4.

## 0. Preparar o runtime

O Google Drive guarda os **dados** entre as aulas, mas o runtime do Colab é temporário.
Programas instalados no runtime podem desaparecer quando a sessão termina.

Por isso, quando uma aula precisar de ferramentas externas, começaremos verificando se
o Conda já existe. Se não existir, ele será instalado antes de qualquer outra configuração.

> Esta deve ser a primeira célula executável do notebook, porque a instalação do Conda
> pode reiniciar o runtime.

In [ ]:
import shutil

if shutil.which("conda"):
    print("Conda já está disponível neste runtime.")
else:
    !pip install -q condacolab
    import condacolab
    condacolab.install()

### Verificar o Conda e configurar Bioconda

Usaremos a configuração recomendada pelo Bioconda: `conda-forge` com maior prioridade,
seguido de `bioconda`, e prioridade estrita.

Como `conda config --add` adiciona canais do menor para o maior nível de prioridade,
executamos primeiro `bioconda` e depois `conda-forge`.

In [ ]:
!conda --version
!conda config --remove-key channels 2>/dev/null || true
!conda config --add channels bioconda
!conda config --add channels conda-forge
!conda config --set channel_priority strict
!conda config --show channels

## 1. Retomar o projeto no Google Drive

Todas as práticas usam a mesma raiz:

`/content/drive/MyDrive/Bioinformatica_Biologia_Molecular`

Os resultados de uma aula são lidos pela aula seguinte. Assim, os **dados persistem**
mesmo quando o runtime do Colab é encerrado.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

ROOT = Path("/content/drive/MyDrive/Bioinformatica_Biologia_Molecular")
RUN = "SRR15736591"
SAMPLE = "hypochilus_petrunkevitchi_SRR15736591"

PASTAS = {
    "01_bancos": ROOT / "01_bancos",
    "02_blast": ROOT / "02_blast",
    "03_raw": ROOT / "03_sra_fastq" / "raw-fastq",
    "04_qc": ROOT / "04_qc_trimming",
    "04_trimmed": ROOT / "04_qc_trimming" / "trimmed",
    "05_assemblies": ROOT / "05_spades" / "spades-assemblies",
    "05_contigs": ROOT / "05_spades" / "spades-assemblies" / "contigs",
    "06_match": ROOT / "06_uce_match",
    "06_probes": ROOT / "06_uce_match" / "probes",
    "06_results": ROOT / "06_uce_match" / "uce-search-results",
    "07_taxon_sets": ROOT / "07_uce_extract" / "taxon-sets" / "all",
    "08_integracao": ROOT / "08_integracao",
    "ambientes": ROOT / "ambientes",
}

for pasta in PASTAS.values():
    pasta.mkdir(parents=True, exist_ok=True)

os.chdir(ROOT)

print("Diretório atual:", Path.cwd())
print("\nEstrutura principal do projeto:")
for chave, pasta in PASTAS.items():
    print(f"{chave:15s} -> {pasta.relative_to(ROOT)}")

## 2. Verificar os FASTQ processados

In [ ]:
TRIMMED = PASTAS["04_trimmed"]
ASSEMBLIES = PASTAS["05_assemblies"]
CONTIGS_DIR = PASTAS["05_contigs"]

R1 = TRIMMED / f"{SAMPLE}_R1_paired.fastq.gz"
R2 = TRIMMED / f"{SAMPLE}_R2_paired.fastq.gz"

if not R1.exists() or not R2.exists():
    raise FileNotFoundError(
        "Reads paired processados não encontrados. Execute primeiro a Aula 4."
    )

OUT = ASSEMBLIES / SAMPLE

print("R1:", R1)
print("R2:", R2)
print("Saída da montagem:", OUT)

## 3. Preparar o ambiente e instalar SPAdes

In [ ]:
!conda env list | grep -qE '^bioinfo[[:space:]]' || conda create -y -n bioinfo python=3.11
!conda install -y -n bioinfo spades
!conda run -n bioinfo spades.py --version

## 4. Executar a montagem

In [ ]:
import shutil

if OUT.exists():
    shutil.rmtree(OUT)

!conda run -n bioinfo spades.py   --only-assembler   --careful   -1 "$R1"   -2 "$R2"   -o "$OUT"   -t 2   -m 12

## 5. Calcular estatísticas básicas

In [ ]:
contigs = OUT / "contigs.fasta"

def fasta_lengths(path):
    lengths, seq = [], []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if seq:
                    lengths.append(len("".join(seq)))
                seq = []
            else:
                seq.append(line)
        if seq:
            lengths.append(len("".join(seq)))
    return lengths

def n50(lengths):
    half = sum(lengths) / 2
    acc = 0
    for L in sorted(lengths, reverse=True):
        acc += L
        if acc >= half:
            return L

lens = fasta_lengths(contigs)
print("Número de contigs:", len(lens))
print("Comprimento total:", sum(lens))
print("Maior contig:", max(lens) if lens else 0)
print("N50:", n50(lens) if lens else 0)

## 6. Preparar o diretório `contigs` usado pelo PHYLUCE

O tutorial oficial do PHYLUCE trabalha com um diretório coletivo chamado `contigs`.
Como estamos no Google Drive, copiaremos o FASTA em vez de depender de links simbólicos.

In [ ]:
filtered = OUT / "contigs_min200.fasta"

with open(contigs) as inp, open(filtered, "w") as out:
    header = None
    seq = []

    def flush():
        if header is not None:
            s = "".join(seq)
            if len(s) >= 200:
                out.write(header + "\n")
                for i in range(0, len(s), 80):
                    out.write(s[i:i+80] + "\n")

    for line in inp:
        line = line.rstrip("\n")
        if line.startswith(">"):
            flush()
            header = line
            seq = []
        else:
            seq.append(line.strip())
    flush()

UCE_CONTIGS = CONTIGS_DIR / f"{SAMPLE}.contigs.fasta"
shutil.copy2(filtered, UCE_CONTIGS)

print("FASTA preparado para PHYLUCE:", UCE_CONTIGS)

## 7. Registrar o ambiente

In [ ]:
ENV_FILE = PASTAS["ambientes"] / "aula05_bioinfo.yml"
!conda env export -n bioinfo --from-history > "$ENV_FILE"
print(ENV_FILE)

## Saída para a próxima aula

A Aula 6 procurará UCEs em:

`05_spades/spades-assemblies/contigs/hypochilus_petrunkevitchi_SRR15736591.contigs.fasta`